# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(url)

# Print a summary
md = dataset.metadata
print(f"Dataset: {md.name}")
print(f"Description: {md.description}")
print(f"Published: {md.datePublished}")
print(f"License: {md.license}")
print(f"Cite as: {md.citeAs}")

## 2. Data Overview

Review available record sets, their `@id`, fields and columns.

> **Note:** For this dataset, we dynamically extract and display the record sets, fields, and columns by their `@id`. These `@id` values should be used in all subsequent analysis.

In [ ]:
# List all record sets by their @id and title
print("Available record sets (by @id and name):")
record_sets = dataset.metadata.recordSet
record_sets_ids = []

for rset in record_sets:
    print(f"- @id: {rset['@id']}, name: {rset.get('name', '<none>')}")
    record_sets_ids.append(rset['@id'])

# For each record set, list its fields
for rset in record_sets:
    print(f"\nRecordSet: {rset['@id']}")
    fields = rset.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        # field is a dict with @id and name, perhaps column
        fid = field['@id'] if isinstance(field, dict) else field
        fname = field.get('name') if isinstance(field, dict) else None
        print(f"  Field @id: {fid}, name: {fname}")
        # columns for this field
        # Not always present, but check
        if isinstance(field, dict):
            columns = field.get('column', [])
            if not isinstance(columns, list):
                columns = [columns]
            for col in columns:
                colid = col['@id'] if isinstance(col, dict) else col
                colname = col.get('name') if isinstance(col, dict) else None
                print(f"    Column @id: {colid}, name: {colname}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values from the overview above.

In [ ]:
# For demonstration, let's choose the **first** record set for extraction.
import pprint

if not record_sets_ids:
    raise ValueError("No record sets found in this dataset.")
# Use the first record set as an example.
record_set_id = record_sets_ids[0]
print(f"Example analysis will use RecordSet: {record_set_id}")

# Extract all records for the record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print("Columns in this DataFrame (Field @id used as column names):")
pprint.pprint(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps. For illustration, we'll:
- Filter on a numeric field
- Normalize the numeric field
- Group by a categorical field (if present)

**All analysis will use columns referred by their Field `@id`.**

In [ ]:
# Identify usable numeric fields by their @ids in the DataFrame
numeric_field_candidates = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_candidates.append(col)

if not numeric_field_candidates:
    print("No numeric fields detected for numeric analysis.")
else:
    print(f"Numeric fields available: {numeric_field_candidates}")
# Choose first numeric field for demo
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None

if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Attempt to group by a likely categorical field
    group_field = None
    for col in df.columns:
        if (pd.api.types.is_object_dtype(df[col]) or df[col].nunique() < 10) and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization

Visualize distributions and relationships for the numeric and categorical fields identified above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

- We loaded and explored the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using `mlcroissant`, referencing record sets, fields, and columns exclusively by their `@id` identifiers.
- Core data fields were programmatically listed from metadata, and analysis was performed in a reproducible, metadata-aware way.
- This pattern supports robust, reusable workflows on Croissant datasets in FAIR repositories.